# Инференс финальной модели, сохранённой в директории model, на тренировочных/валидационных/тестовых данных, расчёт финальных метрик

### Назначение: 
Инференс обученной модели.
### Содержимое:
- Запуск модели, сохранённой в директории model/.
- Реализация скрипта/модуля таким образом, чтобы организаторы могли запустить ваш код в изолированном окружении.
- Расчёт финальных метрик:
    - ROC-AUC на паре ПТСР/контроль с бутстрэп-ДИ (2000 итераций);
    - сбалансированная точность;
    - специфичность на группах «нормальное старение» и «соматоформные расстройства» (доля ложноположительных ПТСР при пороге 0.5);
    - метрики отдельно по возрастным подгруппам (там, где возраст известен).
- Формирование выходного CSV с колонками subject_id, ptsd_probability.

### Виртуальная среда и установка кернела

`conda create -n check-env python=3.12.14 pip -y`

`conda activate check-env`

`python -m pip install -r requirements.txt`

`conda activate check-env`

`python -m ipykernel install --user --name check-env --display-name "Python (check-env)"`

### Импорты

In [13]:
from pathlib import Path
import sys
import joblib
import pandas as pd
from sklearn.metrics import roc_auc_score
import subprocess

RAW_DIR = Path("../data/neurodiagnostics-train")
PREP_DIR = Path("../model/neurodiagnostics-train-prep-pipeline")
FEATURES_DIR = Path("../model/pipeline_features")

LOSO_EEG_CSV = Path("../results/pipeline_loso_eeg.csv")
LOSO_EEG_BEHAV_CSV = Path("../results/pipeline_loso_eeg_behavioral.csv")
MODEL_EEG = Path("../model/model_eeg.joblib")
LOSO_EEG_CSV = Path("results/loso_eeg.csv")

MODEL_EEG_BEHAV = Path("../model/model_eeg_behav.joblib")
LOSO_EEG_BEHAV_CSV = Path("results/loso_eeg_behav.csv")
N_JOBS = 10

### Запуск предсказания - итоги сохраняются в results.csv

In [14]:
import sys
sys.path.append("../model")

import pipeline_data_preparation as dp
import pipeline_feature_extraction as fex
import pipeline_model_training as mt
model = joblib.load(MODEL_EEG)

In [ ]:
subject_dir = RAW_DIR / "ПТСР"    # проверяющим - поменяйте на свою

../data/neurodiagnostics-train/ПТСР/UIPO -> P(PTSD) = 0.7172487127831727


In [35]:
# предсказание
example_raw_dir = next((subject_dir).iterdir())
dp.predict(model, example_raw_dir)   
dp.run(model, RAW_DIR, "results.csv") 
pd.read_csv("results.csv").head()

scoring raw subjects:  93%|█████████▎| 155/166 [01:20<00:04,  2.29subject/s][FAIL] XCFU5: Bad EDF file provided.
Traceback (most recent call last):
  File "/home/veronika/miniconda3/envs/check-env/lib/python3.12/site-packages/mne/io/edf/edf.py", line 1151, in _read_edf_header
    header_nbytes = int(_edf_str(fid.read(8)))
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^
ValueError: invalid literal for int() with base 10: ''

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/veronika/neurotone/atacheskaya-panika/results/../model/pipeline_data_preparation.py", line 88, in run
    except Exception as e:
                ^^^^^^^^^^^
ValueError: Bad EDF file provided.
scoring raw subjects: 100%|██████████| 166/166 [01:25<00:00,  1.94subject/s]

wrote 165 predictions -> results.csv


,subject_id,ptsd_probability
0,КС010_19,0.734764
1,КС011_25,0.851627
2,КС012_58,0.745080
3,КС013_23,0.744527
4,КС014_22,0.574522


###  Проверка работоспособности

In [19]:
example_raw_dir = next((RAW_DIR / "ПТСР").iterdir())
dp.predict(model, example_raw_dir)   
dp.run(model, RAW_DIR, "sanity.csv") 
pd.read_csv("sanity.csv").head()


scoring raw subjects:  93%|█████████▎| 155/166 [01:41<00:13,  1.19s/subject][FAIL] XCFU5: Bad EDF file provided.
Traceback (most recent call last):
  File "/home/veronika/miniconda3/envs/check-env/lib/python3.12/site-packages/mne/io/edf/edf.py", line 1151, in _read_edf_header
    header_nbytes = int(_edf_str(fid.read(8)))
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^
ValueError: invalid literal for int() with base 10: ''

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/veronika/neurotone/atacheskaya-panika/results/../model/pipeline_data_preparation.py", line 88, in run
    except Exception as e:
                ^^^^^^^^^^^
ValueError: Bad EDF file provided.
scoring raw subjects: 100%|██████████| 166/166 [01:49<00:00,  1.52subject/s]

wrote 165 predictions -> sanity.csv


,subject_id,ptsd_probability
0,КС010_19,0.734764
1,КС011_25,0.851627
2,КС012_58,0.745080
3,КС013_23,0.744527
4,КС014_22,0.574522


### Просмотр результатов обучения

In [30]:
loso_df = pd.read_csv("results/loso_eeg.csv")
loso_behav_df = pd.read_csv("results/loso_eeg_behav.csv")
  # -- ROC-AUC + bootstrap CI на паре ПТСР/Норма
  # -- сбалансированная точность
  # -- специфичность на Соматоформных (и "нормальном старении", если появится)
  # -- метрики по возрастным подгруппам
  # -- сравнение loso_df vs loso_behav_df (обязательная абляция по поведенческим)

loso_df, loso_behav_df

(     fold subject_id          group  true_label  ptsd_probability
 0       0   КС010_19          Норма           0          0.746552
 1       1   КС011_25          Норма           0          0.866748
 2       2   КС012_58          Норма           0          0.762532
 3       3   КС013_23          Норма           0          0.752890
 4       4   КС014_22          Норма           0          0.590446
 ..    ...        ...            ...         ...               ...
 160   160      ZAQH1  Соматоформные           0          0.050726
 161   161      ZILO6  Соматоформные           0          0.067659
 162   162      ZJLO6  Соматоформные           0          0.348587
 163   163      ZWAX5  Соматоформные           0          0.532978
 164   164      ZXCV1  Соматоформные           0          0.750807
 
 [165 rows x 5 columns],
      fold subject_id          group  true_label  ptsd_probability
 0       0   КС010_19          Норма           0          0.711776
 1       1   КС011_25          Норм